In [2]:
from glob import glob
import pandas as pd

In [3]:
experimentsListFolder = glob('OptimizeIterationsResults/OptunaDDPGExpBase*')
experimentsListFolder

['OptimizeIterationsResults\\OptunaDDPGExpBase_CorrelationPunishment05',
 'OptimizeIterationsResults\\OptunaDDPGExpBase_CorrelationPunishment07',
 'OptimizeIterationsResults\\OptunaDDPGExpBase_RewardNetProfit',
 'OptimizeIterationsResults\\OptunaDDPGExpBase_RewardNetProfit_ShuffleDaysTrue',
 'OptimizeIterationsResults\\OptunaDDPGExpBase_RMemorySize10',
 'OptimizeIterationsResults\\OptunaDDPGExpBase_RMemorySize3',
 'OptimizeIterationsResults\\OptunaDDPGExpBase_RMemorySize3_NetProfit',
 'OptimizeIterationsResults\\OptunaDDPGExpBase_ShuffleDaysTrue',
 'OptimizeIterationsResults\\OptunaDDPGExpBase_Total']

In [4]:
path = 'OptimizeIterationsResults/OptunaDDPGExpBase_Total/000000\\2024-10-26_00-46-07_218\\metadata.pkl'

In [5]:
folder = 'OptimizeIterationsResults/OptunaDDPGExpBase_Total'

In [6]:
folder.split('/OptunaDDPGExpBase_')[-1] 

'Total'

In [7]:
def getTrainData(folder):
    # return the OptimizationDataResume
    optimizationPath = f'{folder}/trialsData.csv'
    opt_df = pd.read_csv(optimizationPath)
    opt_df['duration'] = pd.to_timedelta(opt_df['duration'])
    opt_df['duration'] = opt_df['duration'].dt.total_seconds() / 60

    metadataPathList = glob(f'{folder}/*/*/metadata.pkl')
    trainingTimes = []
    testingTimes = []
    for mdPath in metadataPathList:
        jsonStruct = pd.read_pickle(mdPath)
        # get the training data
        trainingTimes.append(jsonStruct['TRAINING_TIME'])
        # get the testing data
        testingTimes.append(jsonStruct['TESTING_TIME'])

    # calculate the mean and std of the training and testing times
    trainingTimes = pd.Series(trainingTimes)
    testingTimes = pd.Series(testingTimes)
    trainingMean = trainingTimes.mean()
    trainingStd = trainingTimes.std()
    testingMean = testingTimes.mean()
    testingStd = testingTimes.std()
    # get total time for training and testing
    totalTrainingTime = trainingTimes.sum()
    totalTestingTime = testingTimes.sum()
    totalTime = totalTrainingTime + totalTestingTime
    
    #number of trials, 
    # mean optimization duration in minutes
    # mean training time in minutes
    # std training time in minutes
    # mean testing time in minutes
    # std testing time in minutes
    # mean total time in minutes
    # mean training time in minutes
    # mean testing time in minutes
    # mean total time in minutes

    return len(opt_df), \
            opt_df['duration'].mean(), \
            trainingMean, \
            trainingStd, \
            testingMean, \
            testingStd, \
            totalTrainingTime, \
            totalTestingTime, \
            totalTime

In [8]:
# creata a dataframe with the columns
DataFinal = []
# get the data for each folder
for folder in experimentsListFolder:
    trials, opt_duration, train_mean, train_std, test_mean, test_std, total_train_time, total_test_time, total_time = getTrainData(folder)
    DataFinal.append([folder.split('\OptunaDDPGExpBase_')[-1],
        trials,
        opt_duration,
        train_mean,
        train_std,
        test_mean,
        test_std,
        total_train_time,
        total_test_time,
        total_time])

In [9]:
columns = ['Exp', 'trials', 'opt_duration', 'train_mean', 'train_std', 'test_mean', 'test_std', 'total_train_time', 'total_test_time', 'total_time']
df_Final = pd.DataFrame(data=DataFinal, columns=columns)

In [10]:
# Convert the columns to 2 decimal places
df_Final = df_Final.round(2)
df_Final

,Exp,trials,opt_duration,train_mean,train_std,test_mean,test_std,total_train_time,total_test_time,total_time
0,CorrelationPunishment05,11,26.73,23.30,2.36,0.42,0.03,2330.43,41.74,2372.17
1,CorrelationPunishment07,18,23.02,18.95,1.25,0.43,0.03,1895.30,42.96,1938.26
2,RewardNetProfit,12,20.84,19.03,1.42,0.42,0.02,1902.59,42.45,1945.03
3,RewardNetProfit_ShuffleDaysTrue,12,23.70,30.50,4.37,0.41,0.03,3050.31,41.12,3091.43
4,RMemorySize10,17,19.71,18.80,1.98,0.43,0.04,1860.76,42.98,1903.74
5,RMemorySize3,19,22.67,22.72,2.63,0.41,0.03,2272.33,40.79,2313.12
6,RMemorySize3_NetProfit,13,21.54,22.78,1.94,0.41,0.03,2278.21,40.81,2319.02
7,ShuffleDaysTrue,12,23.70,23.02,1.93,0.43,0.02,2301.98,42.78,2344.76
8,Total,14,24.57,19.76,1.29,0.43,0.02,1975.78,42.84,2018.62


In [11]:
df_Final['Opt_Duration'] = df_Final['trials'] * df_Final['opt_duration']

In [15]:
# Pass the columns to hours
df_Final.columns

Index(['Exp', 'trials', 'opt_duration', 'train_mean', 'train_std', 'test_mean',
       'test_std', 'total_train_time', 'total_test_time', 'total_time',
       'Opt_Duration'],
      dtype='object')

In [16]:
df_FinalFilt = df_Final[['Exp','Opt_Duration','total_train_time','total_test_time','total_time']]

In [17]:
df_FinalFilt

,Exp,Opt_Duration,total_train_time,total_test_time,total_time
0,CorrelationPunishment05,294.03,2330.43,41.74,2372.17
1,CorrelationPunishment07,414.36,1895.30,42.96,1938.26
2,RewardNetProfit,250.08,1902.59,42.45,1945.03
3,RewardNetProfit_ShuffleDaysTrue,284.40,3050.31,41.12,3091.43
4,RMemorySize10,335.07,1860.76,42.98,1903.74
5,RMemorySize3,430.73,2272.33,40.79,2313.12
6,RMemorySize3_NetProfit,280.02,2278.21,40.81,2319.02
7,ShuffleDaysTrue,284.40,2301.98,42.78,2344.76
8,Total,343.98,1975.78,42.84,2018.62


In [18]:
df_FinalFilt[['Opt_Duration','total_train_time','total_test_time','total_time']] \
 = df_FinalFilt[['Opt_Duration','total_train_time','total_test_time','total_time']] / 60

C:\Users\tcorn\AppData\Local\Temp\ipykernel_11924\1250479259.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_FinalFilt[['Opt_Duration','total_train_time','total_test_time','total_time']] \


In [20]:
df_FinalFilt.round(2)

,Exp,Opt_Duration,total_train_time,total_test_time,total_time
0,CorrelationPunishment05,4.90,38.84,0.70,39.54
1,CorrelationPunishment07,6.91,31.59,0.72,32.30
2,RewardNetProfit,4.17,31.71,0.71,32.42
3,RewardNetProfit_ShuffleDaysTrue,4.74,50.84,0.69,51.52
4,RMemorySize10,5.58,31.01,0.72,31.73
5,RMemorySize3,7.18,37.87,0.68,38.55
6,RMemorySize3_NetProfit,4.67,37.97,0.68,38.65
7,ShuffleDaysTrue,4.74,38.37,0.71,39.08
8,Total,5.73,32.93,0.71,33.64


In [21]:
df_FinalFilt['TotalTotal'] = df_FinalFilt['Opt_Duration'] + df_FinalFilt['total_train_time'] + df_FinalFilt['total_test_time']
df_FinalFilt['TotalTotal'] = df_FinalFilt['TotalTotal'].round(2)
df_FinalFilt

C:\Users\tcorn\AppData\Local\Temp\ipykernel_11924\2611086261.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_FinalFilt['TotalTotal'] = df_FinalFilt['Opt_Duration'] + df_FinalFilt['total_train_time'] + df_FinalFilt['total_test_time']
C:\Users\tcorn\AppData\Local\Temp\ipykernel_11924\2611086261.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_FinalFilt['TotalTotal'] = df_FinalFilt['TotalTotal'].round(2)


,Exp,Opt_Duration,total_train_time,total_test_time,total_time,TotalTotal
0,CorrelationPunishment05,4.900500,38.840500,0.695667,39.536167,44.44
1,CorrelationPunishment07,6.906000,31.588333,0.716000,32.304333,39.21
2,RewardNetProfit,4.168000,31.709833,0.707500,32.417167,36.59
3,RewardNetProfit_ShuffleDaysTrue,4.740000,50.838500,0.685333,51.523833,56.26
4,RMemorySize10,5.584500,31.012667,0.716333,31.729000,37.31
5,RMemorySize3,7.178833,37.872167,0.679833,38.552000,45.73
6,RMemorySize3_NetProfit,4.667000,37.970167,0.680167,38.650333,43.32
7,ShuffleDaysTrue,4.740000,38.366333,0.713000,39.079333,43.82
8,Total,5.733000,32.929667,0.714000,33.643667,39.38


In [25]:
df_FinalFilt['TotalTotal'].sum() / 24

16.085833333333333